---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 10 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas9_[NIM]_[Nama Lengkap].ipynb`**, koneksikan ke MongoDB, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas9_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan MongoDB (`sudo systemctl status mongod`) aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Create | Minimal 12 dokumen tersimpan dengan field sesuai ketentuan kategori | 20% |
| B. Read | Ketiga kueri menghasilkan output yang tepat sesuai instruksi | 20% |
| C. Update | Field `status` berhasil ditambahkan & diperbarui sesuai instruksi | 15% |
| D. Integrasi Spark | Konversi ke Spark DataFrame berhasil; analisis `groupBy` benar | 25% |
| E. Kesimpulan | Refleksi menunjukkan pemahaman konseptual yang tepat tentang tantangan skema | 20% |


## Bagian A. Membangun Katalog (Create)

In [2]:
from pymongo import MongoClient

# 1. Koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017/")

# 2. Membuat database 'kopi_nusantara' dan collection 'menu'
db_kopi = client["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]

# Bersihkan collection terlebih dahulu jika sudah ada data lama
koleksi_menu.delete_many({})

# 3. Data 12 dokumen menu (Kopi, Makanan, dan kategori tambahan)
data_menu = [
    # Kategori Kopi (Wajib: nama_menu, harga, kategori, tingkat_kepahitan)
    {"nama_menu": "Espresso Single", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 4, "stok": 40},
    {"nama_menu": "Kopi Tubruk Tuban", "kategori": "Kopi", "harga": 15000, "tingkat_kepahitan": 5, "stok": 60},
    {"nama_menu": "Caffè Latte", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 35},
    {"nama_menu": "Piccolo Latte", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 3, "stok": 30},
    
    # Kategori Makanan (Wajib: nama_menu, harga, kategori, pedas_level)
    {"nama_menu": "Nasi Goreng Rempah", "kategori": "Makanan", "harga": 32000, "pedas_level": 2, "rekomendasi_barista": True},
    {"nama_menu": "Mie Goreng Jawa", "kategori": "Makanan", "harga": 28000, "pedas_level": 3, "stok": 25},
    {"nama_menu": "Ayam Geprek Sambal Matah", "kategori": "Makanan", "harga": 35000, "pedas_level": 3, "rekomendasi_barista": True},
    {"nama_menu": "Roti Bakar Cokelat", "kategori": "Makanan", "harga": 20000, "pedas_level": 0, "stok": 45},
    
    # Kategori Non-Kopi & Camilan (Variasi Atribut Tambahan)
    {"nama_menu": "Matcha Latte", "kategori": "Non-Kopi", "harga": 30000, "stok": 20},
    {"nama_menu": "Croissant Original", "kategori": "Snack", "harga": 22000, "stok": 15},
    {"nama_menu": "Singkong Goreng Keju", "kategori": "Snack", "harga": 18000, "stok": 40}
]

# Insert 12 dokumen
koleksi_menu.insert_many(data_menu)
print(f"Berhasil menyisipkan {koleksi_menu.count_documents({})} dokumen ke collection 'menu'.")

Berhasil menyisipkan 12 dokumen ke collection 'menu'.


## Bagian B. Kueri Bertingkat (Read)

In [3]:
# 1. Kopi dengan tingkat_kepahitan > 3
print("=== 1. Menu Kopi dengan tingkat_kepahitan > 3 ===")
kueri_1 = koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}})
for menu in kueri_1:
    print(f"- {menu['nama_menu']} (Kepahitan: {menu['tingkat_kepahitan']})")

print("\n=== 2. Menu dengan harga antara 15.000 dan 30.000 ===")
kueri_2 = koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}})
for menu in kueri_2:
    print(f"- {menu['nama_menu']} (Rp{menu['harga']:,})")

print("\n=== 3. Menu yang TIDAK memiliki field 'pedas_level' ===")
kueri_3 = koleksi_menu.find({"pedas_level": {"$exists": False}})
for menu in kueri_3:
    print(f"- {menu['nama_menu']} [{menu['kategori']}]")

=== 1. Menu Kopi dengan tingkat_kepahitan > 3 ===
- Espresso Single (Kepahitan: 5)
- Americano (Kepahitan: 4)
- Kopi Tubruk Tuban (Kepahitan: 5)

=== 2. Menu dengan harga antara 15.000 dan 30.000 ===
- Espresso Single (Rp18,000)
- Americano (Rp22,000)
- Kopi Tubruk Tuban (Rp15,000)
- Caffè Latte (Rp28,000)
- Piccolo Latte (Rp25,000)
- Mie Goreng Jawa (Rp28,000)
- Roti Bakar Cokelat (Rp20,000)
- Matcha Latte (Rp30,000)
- Croissant Original (Rp22,000)
- Singkong Goreng Keju (Rp18,000)

=== 3. Menu yang TIDAK memiliki field 'pedas_level' ===
- Espresso Single [Kopi]
- Americano [Kopi]
- Kopi Tubruk Tuban [Kopi]
- Caffè Latte [Kopi]
- Piccolo Latte [Kopi]
- Matcha Latte [Non-Kopi]
- Croissant Original [Snack]
- Singkong Goreng Keju [Snack]


## Bagian C. Pembaruan Data (Update)

In [4]:
# 1. Tambahkan field status: "Tersedia" ke seluruh dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
print("Field 'status' bernilai 'Tersedia' berhasil ditambahkan ke seluruh dokumen.")

# 2. Ubah status menjadi "Habis" khusus untuk 2 menu pilihan
koleksi_menu.update_one({"nama_menu": "Croissant Original"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Matcha Latte"}, {"$set": {"status": "Habis"}})
print("Status 2 menu pilihan berhasil diubah menjadi 'Habis'.\n")

# Verifikasi tampilan status
print("=== Tampilan Status Menu Terbaru ===")
for menu in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"- {menu['nama_menu']}: {menu['status']}")

Field 'status' bernilai 'Tersedia' berhasil ditambahkan ke seluruh dokumen.
Status 2 menu pilihan berhasil diubah menjadi 'Habis'.

=== Tampilan Status Menu Terbaru ===
- Espresso Single: Tersedia
- Americano: Tersedia
- Kopi Tubruk Tuban: Tersedia
- Caffè Latte: Tersedia
- Piccolo Latte: Tersedia
- Nasi Goreng Rempah: Tersedia
- Mie Goreng Jawa: Tersedia
- Ayam Geprek Sambal Matah: Tersedia
- Roti Bakar Cokelat: Tersedia
- Matcha Latte: Habis
- Croissant Original: Habis
- Singkong Goreng Keju: Tersedia


## Bagian D. Integrasi ke Spark

In [10]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Ambil dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_menu.find({}))

# 2. Konversi ke pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# ----------------------------------------------------
# PERBAIKAN TIPE DATA PANDAS SEBELUM MASUK SPARK
# ----------------------------------------------------
# Ubah _id ke string
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Ubah NaN/None pada kolom numerik bertipe opsional agar tidak membuat error
# Gantilah NaN dengan None standar Python agar PySpark menganggapnya sebagai nullable
df_pandas = df_pandas.replace({np.nan: None})
# ----------------------------------------------------

# 3. Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# 4. Tampilkan printSchema()
print("=== 1. Schema Spark DataFrame ===")
df_spark_menu.printSchema()

# 5. Tampilkan Rata-rata Harga per Kategori
from pyspark.sql.functions import avg, round

print("=== 2. Rata-rata Harga per Kategori ===")
df_spark_menu.groupBy("kategori").agg(
    round(avg("harga"), 2).alias("rata_rata_harga")
).show()

=== 1. Schema Spark DataFrame ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)

=== 2. Rata-rata Harga per Kategori ===


+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|        21600.0|
| Makanan|        28750.0|
|Non-Kopi|        30000.0|
|   Snack|        20000.0|
+--------+---------------+



Bagian E. Kesimpulan

### E. Kesimpulan & Refleksi

Tantangan utama yang dihadapi saat mengonversi data dari MongoDB ke Spark DataFrame adalah adanya perbedaan sifat skema antara kedua platform. MongoDB menggunakan skema fleksibel (*schema-less*), di mana setiap dokumen dapat memiliki struktur dan field yang tidak seragam (misalnya, field `tingkat_kepahitan` hanya ada pada kategori Kopi, sedangkan `pedas_level` hanya ada pada kategori Makanan). Sebaliknya, Spark DataFrame mewajibkan struktur tabel dengan skema yang seragam untuk seluruh baris data. 

Saat data dari MongoDB dikonversi ke Spark DataFrame (melalui Pandas), tantangan ini diatasi dengan pengisian nilai `null` (*None*) secara otomatis oleh sistem pada kolom-kolom yang tidak dimiliki oleh dokumen tertentu. Langkah penanganan ini memastikan bahwa data berstruktur dinamis dari NoSQL dapat dipetakan secara konsisten menjadi bentuk tabular tanpa kehilangan informasi penting, sehingga analisis seperti agregasi `groupBy` tetap dapat berjalan dengan lancar tanpa error.